# 03 - 学习率调度器（LR Schedulers）

## 学习目标

1. 理解学习率调度器的基类和核心概念
2. 掌握常用调度器的使用方法和适用场景
3. 能够可视化不同调度器的学习率曲线
4. 掌握 warmup + cosine decay 的组合策略

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import (
    StepLR, MultiStepLR, ExponentialLR, CosineAnnealingLR,
    ReduceLROnPlateau, CosineAnnealingWarmRestarts,
    OneCycleLR, LambdaLR,
)

## 1. 学习率调度器概述

### 为什么需要调整学习率？

- **训练初期**：较大学习率，快速找到大方向
- **训练后期**：较小学习率，精细调整，避免震荡

### _LRScheduler 基类

所有调度器继承自 `_LRScheduler`，核心属性和方法：

| 属性/方法 | 说明 |
|----------|------|
| `optimizer` | 关联的优化器 |
| `base_lrs` | 初始学习率列表（每个参数组一个） |
| `last_epoch` | 上一次调用 step() 的 epoch |
| `step()` | 更新学习率 |
| `get_last_lr()` | 获取当前学习率 |

### 使用方式

```python
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=10, gamma=0.1)

for epoch in range(100):
    train(...)           # 训练一个 epoch
    scheduler.step()     # 更新学习率
```

In [ ]:
# 辅助函数：收集调度器的学习率变化
def collect_lrs(scheduler_fn, total_epochs=100):
    """创建调度器并收集学习率变化。

    Args:
        scheduler_fn: 接受 optimizer 参数并返回 scheduler 的函数
        total_epochs: 总 epoch 数

    Returns:
        学习率列表
    """
    model = nn.Linear(1, 1)
    optimizer = optim.SGD(model.parameters(), lr=0.1)
    scheduler = scheduler_fn(optimizer)
    
    lrs = []
    for epoch in range(total_epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        # 模拟训练步骤（ReduceLROnPlateau 需要特殊处理）
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(1.0)  # 传入指标值
        else:
            scheduler.step()
    return lrs

print("辅助函数已定义")

## 2. StepLR：固定步长衰减

每隔 `step_size` 个 epoch，学习率乘以 `gamma`。

$$\text{lr}_{epoch} = \text{lr}_0 \times \gamma^{\lfloor epoch / step\_size \rfloor}$$

In [ ]:
# StepLR 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

print("StepLR (step_size=10, gamma=0.5):")
print(f"  初始 lr: {optimizer.param_groups[0]['lr']}")

for epoch in range(50):
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch + 1}: lr = {optimizer.param_groups[0]['lr']:.6f}")

print("\n每 10 个 epoch，lr 减半")

## 3. MultiStepLR：里程碑衰减

在指定的 epoch（里程碑）处衰减学习率。

比 StepLR 更灵活，可以在不同阶段使用不同的衰减时机。

In [ ]:
# MultiStepLR 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = MultiStepLR(optimizer, milestones=[30, 60, 80], gamma=0.1)

print("MultiStepLR (milestones=[30, 60, 80], gamma=0.1):")
key_epochs = [0, 29, 30, 59, 60, 79, 80, 99]

for epoch in range(100):
    if epoch in key_epochs:
        print(f"  Epoch {epoch}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()

print("\n在 epoch 30, 60, 80 处学习率各降 10 倍")

## 4. ExponentialLR：指数衰减

每个 epoch 学习率乘以 `gamma`。

$$\text{lr}_{epoch} = \text{lr}_0 \times \gamma^{epoch}$$

In [ ]:
# ExponentialLR 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = ExponentialLR(optimizer, gamma=0.95)

print("ExponentialLR (gamma=0.95):")
for epoch in range(0, 100, 10):
    # 需要先输出，再 step
    if epoch == 0:
        print(f"  Epoch {epoch}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()
    if epoch > 0 and epoch % 10 == 0:
        print(f"  Epoch {epoch}: lr = {optimizer.param_groups[0]['lr']:.6f}")

# 重新创建并展示
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = ExponentialLR(optimizer, gamma=0.95)

print("\n完整的关键 epoch lr:")
for epoch in range(100):
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()

print("\n指数衰减: 每个 epoch lr 乘以 0.95")

## 5. CosineAnnealingLR：余弦退火

学习率按照余弦函数从初始值衰减到 `eta_min`。

$$\text{lr}_t = \eta_{min} + \frac{1}{2}(\eta_{max} - \eta_{min})(1 + \cos(\frac{t}{T_{max}} \pi))$$

特点：平滑衰减，训练末期学习率接近 0

In [ ]:
# CosineAnnealingLR 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = CosineAnnealingLR(optimizer, T_max=50, eta_min=0.001)

print("CosineAnnealingLR (T_max=50, eta_min=0.001):")
for epoch in range(50):
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()

print("\n余弦退火: 平滑从 0.1 下降到 0.001")

## 6. ReduceLROnPlateau：自适应调整

监控一个指标（如验证集 loss），当指标不再改善时降低学习率。

**注意**: `step()` 需要传入监控的指标值，而不是自动调用。

```python
scheduler.step(val_loss)  # 需要传入指标！
```

In [ ]:
# ReduceLROnPlateau 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = ReduceLROnPlateau(
    optimizer,
    mode='min',        # 监控指标越小越好
    factor=0.5,        # 衰减因子
    patience=5,        # 连续 5 次不改善才衰减
    min_lr=1e-6,       # 最小学习率
    verbose=True,      # 打印信息
)

# 模拟验证 loss：先下降，后趋于平稳
val_losses = (
    [1.0, 0.9, 0.8, 0.7, 0.6, 0.55, 0.52, 0.50, 0.49, 0.48]
    + [0.48, 0.48, 0.47, 0.47, 0.47, 0.47]  # 平台期
    + [0.45, 0.44, 0.43, 0.43, 0.43, 0.43, 0.43, 0.43, 0.43, 0.43]
)

print("ReduceLROnPlateau (patience=5, factor=0.5):\n")
for epoch, val_loss in enumerate(val_losses):
    lr = optimizer.param_groups[0]['lr']
    print(f"  Epoch {epoch:2d}: val_loss={val_loss:.2f}, lr={lr:.6f}")
    scheduler.step(val_loss)

## 7. CosineAnnealingWarmRestarts：带重启的余弦退火

学习率周期性地重启，每次重启后从较大的学习率开始。

- `T_0`：第一个周期的 epoch 数
- `T_mult`：每次重启后周期长度的乘数

In [ ]:
# CosineAnnealingWarmRestarts 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=0.001)

print("CosineAnnealingWarmRestarts (T_0=10, T_mult=2):")
print("  第 1 个周期: 10 epochs")
print("  第 2 个周期: 20 epochs")
print("  第 3 个周期: 40 epochs\n")

for epoch in range(70):
    if epoch % 5 == 0:
        print(f"  Epoch {epoch:3d}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()

print("\n每个周期结束时学习率重启到初始值")

## 8. OneCycleLR：一周期策略

超级收敛策略：学习率先升后降，只执行一个完整周期。

```
lr: 低 ──> 高 ──> 很低
```

**注意**: OneCycleLR 是按**步**调度的，在每个 batch 后调用 `step()`。

In [ ]:
# OneCycleLR 示例
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.01)  # 初始 lr 较小

epochs = 10
steps_per_epoch = 100  # 每个 epoch 100 个 batch
total_steps = epochs * steps_per_epoch

scheduler = OneCycleLR(
    optimizer,
    max_lr=0.1,            # 最大学习率
    total_steps=total_steps,
    pct_start=0.3,         # 前 30% 的步数用于升温
    anneal_strategy='cos', # 余弦退火
)

print("OneCycleLR (max_lr=0.1, pct_start=0.3):")
lrs = []
for step in range(total_steps):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

# 打印关键节点的 lr
key_points = [0, 100, 200, 300, 500, 700, 900, 999]
for s in key_points:
    print(f"  Step {s:4d} (Epoch {s // 100}): lr = {lrs[s]:.6f}")

print(f"\n最大 lr 出现在 Step {lrs.index(max(lrs))}: {max(lrs):.6f}")
print(f"最小 lr: {min(lrs):.8f}")

## 9. LambdaLR：自定义调度

使用自定义函数来计算学习率乘数。

$$\text{lr}_{epoch} = \text{lr}_0 \times \lambda(epoch)$$

最灵活的调度器，可以实现任意学习率策略。

In [ ]:
# LambdaLR: 实现线性 warmup
model = nn.Linear(1, 1)
optimizer = optim.SGD(model.parameters(), lr=0.1)

warmup_epochs = 10

# warmup: 前 10 个 epoch 线性增长到初始 lr
def warmup_lambda(epoch):
    if epoch < warmup_epochs:
        return epoch / warmup_epochs  # 线性增长
    return 1.0  # 之后保持不变

scheduler = LambdaLR(optimizer, lr_lambda=warmup_lambda)

print("LambdaLR (线性 warmup, 10 epochs):")
for epoch in range(20):
    print(f"  Epoch {epoch:2d}: lr = {optimizer.param_groups[0]['lr']:.6f}")
    scheduler.step()

In [ ]:
# LambdaLR: 不同参数组使用不同的 lambda
class TwoPartModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Linear(10, 10)
        self.head = nn.Linear(10, 2)
    
    def forward(self, x):
        return self.head(self.backbone(x))

model = TwoPartModel()
optimizer = optim.SGD([
    {'params': model.backbone.parameters(), 'lr': 0.001},
    {'params': model.head.parameters(), 'lr': 0.01},
])

# backbone 缓慢 warmup，head 直接用全量 lr
scheduler = LambdaLR(optimizer, lr_lambda=[
    lambda epoch: min(1.0, epoch / 20),  # backbone: 20 epoch warmup
    lambda epoch: 1.0,                    # head: 不变
])

print("不同参数组的 LambdaLR:")
for epoch in range(0, 25, 5):
    lrs = [g['lr'] for g in optimizer.param_groups]
    print(f"  Epoch {epoch:2d}: backbone_lr={lrs[0]:.6f}, head_lr={lrs[1]:.6f}")
    for _ in range(5):
        scheduler.step()

## 10. 可视化所有调度器的曲线

通过打印学习率值来对比各个调度器的行为。

In [ ]:
# 收集所有调度器的学习率变化
total_epochs = 100

schedulers = {
    'StepLR(30, 0.1)': lambda opt: StepLR(opt, step_size=30, gamma=0.1),
    'MultiStepLR([30,60,80])': lambda opt: MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.1),
    'ExponentialLR(0.95)': lambda opt: ExponentialLR(opt, gamma=0.95),
    'CosineAnnealing(100)': lambda opt: CosineAnnealingLR(opt, T_max=100, eta_min=0.001),
    'CosineWarmRestarts(20,2)': lambda opt: CosineAnnealingWarmRestarts(opt, T_0=20, T_mult=2, eta_min=0.001),
}

all_lrs = {}
for name, sched_fn in schedulers.items():
    all_lrs[name] = collect_lrs(sched_fn, total_epochs)

# 打印关键 epoch 的 lr 对比表
key_epochs = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 99]

print(f"{'Epoch':>6}", end='')
for name in schedulers:
    short_name = name[:20]
    print(f"  {short_name:>22}", end='')
print()
print('-' * (6 + 24 * len(schedulers)))

for epoch in key_epochs:
    print(f"{epoch:6d}", end='')
    for name in schedulers:
        print(f"  {all_lrs[name][epoch]:>22.6f}", end='')
    print()

## 11. 调度器总结表

PyTorch 提供了 14 种学习率调度器：

| 调度器 | 策略 | step() 调用位置 | 适用场景 |
|--------|------|----------------|----------|
| `StepLR` | 固定步长衰减 | 每 epoch | 简单训练 |
| `MultiStepLR` | 里程碑衰减 | 每 epoch | 经验丰富时手动设定 |
| `ExponentialLR` | 指数衰减 | 每 epoch | 平滑衰减 |
| `CosineAnnealingLR` | 余弦退火 | 每 epoch | CV 训练（最常用） |
| `ReduceLROnPlateau` | 自适应衰减 | 每 epoch（传入指标） | 不确定衰减时机 |
| `CosineAnnealingWarmRestarts` | 带重启余弦 | 每 epoch | 长训练、需要逃离局部最优 |
| `OneCycleLR` | 一周期策略 | 每 batch | 超级收敛 |
| `LambdaLR` | 自定义函数 | 每 epoch | 需要自定义策略 |
| `MultiplicativeLR` | 自定义乘法衰减 | 每 epoch | 简单乘法衰减 |
| `LinearLR` | 线性衰减 | 每 epoch | warmup 阶段 |
| `PolynomialLR` | 多项式衰减 | 每 epoch | 平滑衰减 |
| `ConstantLR` | 恒定倍数 | 每 epoch | warmup 阶段 |
| `SequentialLR` | 串联多个调度器 | 每 epoch | 组合策略 |
| `ChainedScheduler` | 链式调度器 | 每 epoch | 组合策略 |

### 选择建议

```
CV 通用 ──> CosineAnnealingLR
Transformer ──> warmup + CosineAnnealingLR（用 LambdaLR 实现）
快速训练 ──> OneCycleLR
不确定衰减时机 ──> ReduceLROnPlateau
长时间训练 ──> CosineAnnealingWarmRestarts
```

## 12. 练习题

### 练习 1：实现 Warmup + Cosine Decay

使用 `LambdaLR` 实现以下策略：
- 前 10 个 epoch：线性 warmup（从 0 到 lr）
- 第 10-100 个 epoch：余弦衰减（从 lr 到 0）

In [ ]:
# 练习 1: Warmup + Cosine Decay
import math

def warmup_cosine_lambda(epoch, warmup_epochs=10, total_epochs=100):
    """Warmup + Cosine Decay 的 lambda 函数。

    Args:
        epoch: 当前 epoch
        warmup_epochs: warmup 阶段的 epoch 数
        total_epochs: 总 epoch 数

    Returns:
        学习率乘数
    """
    # TODO: 实现
    # warmup 阶段: 线性从 0 增长到 1
    # cosine 阶段: 从 1 衰减到 0
    pass

# 测试
# model = nn.Linear(1, 1)
# optimizer = optim.SGD(model.parameters(), lr=0.1)
# scheduler = LambdaLR(optimizer, lr_lambda=lambda e: warmup_cosine_lambda(e))
#
# print("Warmup + Cosine Decay:")
# for epoch in range(0, 100, 5):
#     print(f"  Epoch {epoch:3d}: lr = {optimizer.param_groups[0]['lr']:.6f}")
#     for _ in range(5):
#         scheduler.step()

### 练习 2：比较不同调度器在训练中的效果

在一个简单的回归问题上，比较不同学习率调度策略对训练 loss 的影响。

In [ ]:
# 练习 2: 比较调度器的训练效果
torch.manual_seed(42)

# 生成简单的回归数据
X = torch.randn(200, 5)
y = X @ torch.tensor([1.0, -2.0, 3.0, -1.0, 0.5]) + 0.1 * torch.randn(200)
y = y.unsqueeze(1)

# TODO: 用不同的调度器训练模型
# 对比: 不用调度器 / StepLR / CosineAnnealingLR
# 打印每 10 个 epoch 的 loss
pass

## 13. 小结

### 核心要点

1. 学习率调度器在训练过程中动态调整学习率
2. 大多数调度器在 **每个 epoch 后** 调用 `step()`
3. `OneCycleLR` 在 **每个 batch 后** 调用 `step()`
4. `ReduceLROnPlateau.step()` 需要 **传入指标值**
5. `CosineAnnealingLR` 是最常用的调度器之一
6. Warmup 可以用 `LambdaLR` 实现

### 常见组合策略

- **CV**: SGD + CosineAnnealingLR
- **NLP/Transformer**: AdamW + Linear Warmup + Cosine Decay
- **快速实验**: Adam + ReduceLROnPlateau

### 注意事项

- `scheduler.step()` 要在 `optimizer.step()` **之后** 调用
- 保存 checkpoint 时也要保存 scheduler 的状态
- 不同调度器的 `step()` 调用频率不同（每 epoch / 每 batch）

### 下一步

接下来我们将把损失函数、优化器、调度器组合起来，实现一个完整的优化实践。